In [21]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import(
roc_auc_score, accuracy_score, precision_recall_fscore_support, classification_report, f1_score, precision_score, recall_score, 
balanced_accuracy_score, confusion_matrix,)



INTERIM_PATH = Path("../data/interim")
PROCESSED_PATH = Path("../data/processed")

feat = pd.read_parquet(INTERIM_PATH / "features_stocks_v2.parquet") 
tgt  = pd.read_parquet(PROCESSED_PATH / "stocks_monthly_targets_splits.parquet")

# keep only needed target cols
tgt = tgt[["date","ticker","excess_log_fwd12","excess_simple_fwd12","label_win_12","split"]]
df  = feat.merge(tgt, on=["date","ticker"], how="inner", validate="one_to_one")

df

,date,ticker,l_stock,l_bench,ex_log,close,volume,mom_log_3,mom_log_6,mom_log_12,...,us10y_lvl,brent_d3m,sector_rel_mom12,gt_ma3,gt_z12,gt_spike,excess_log_fwd12,excess_simple_fwd12,label_win_12,split
0,2005-02-28,AAPL,0.154188,0.018727,0.135461,1.346702,21448946400,NaN,NaN,NaN,...,4.14,-0.078815,NaN,2.666667,2.672612,1.0,0.361082,0.434882,1,train
1,2005-03-31,AAPL,-0.073765,-0.019303,-0.054462,1.250938,14675920000,NaN,NaN,NaN,...,4.36,0.125216,NaN,3.333333,1.963961,0.0,0.316503,0.372320,1,train
2,2005-04-30,AAPL,-0.144597,-0.020314,-0.124284,1.082525,19375518400,NaN,NaN,NaN,...,4.50,0.276100,NaN,3.000000,-0.055641,0.0,0.544076,0.723015,1,train
3,2005-05-31,AAPL,0.097677,0.029512,0.068165,1.193600,12858294400,-0.064175,NaN,NaN,...,4.21,0.128435,NaN,2.333333,0.507093,0.0,0.343768,0.410252,1,train
4,2005-06-30,AAPL,-0.077092,-0.000143,-0.076949,1.105041,11327195600,-0.120685,NaN,NaN,...,4.00,-0.016696,NaN,1.333333,-0.232495,0.0,0.377903,0.459222,1,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2355,2024-05-31,XOM,-0.008576,0.046904,-0.055480,111.359818,415960600,0.149338,0.129301,0.034566,...,4.69,0.061347,0.030324,0.000000,NaN,0.0,-0.216765,-0.194881,0,test
2356,2024-06-30,XOM,-0.010329,0.034082,-0.044411,110.215508,345560000,0.124266,0.150546,0.172815,...,4.51,-0.062955,0.049141,0.000000,NaN,0.0,-0.159233,-0.147202,0,test
2357,2024-07-31,XOM,0.029697,0.011258,0.018439,113.537659,277953800,-0.001592,0.158345,0.105420,...,4.36,0.012570,0.034423,0.000000,NaN,0.0,-0.164106,-0.151348,0,test
2358,2024-08-31,XOM,-0.005496,0.022578,-0.028074,112.915359,284673400,0.010792,0.160130,0.135210,...,4.09,-0.080695,0.093016,0.000000,NaN,0.0,-0.131459,-0.123185,0,test


In [4]:
# Drop the columns that won't be used in the model as features 

id_cols      = {"date","ticker","split"}
target_cols  = {"excess_log_fwd12","excess_simple_fwd12","label_win_12"}
raw_base_cols = {"l_stock","l_bench","ex_log","close","volume"} 

drop_cols = id_cols | target_cols | raw_base_cols

feature_cols = [c for c in df.columns
                if c not in drop_cols and df[c].dtype.kind in "fcbiu"]

# Keep only features that have at least one non-missing value
feature_cols = [c for c in feature_cols if df[c].notna().any()]

print("n_features:", len(feature_cols))
print(sorted(feature_cols)[:10], "…")

n_features: 28
['beta_36', 'brent_d3m', 'country_UK', 'dist_sma_12', 'drawdown_12', 'dxy_lvl', 'gt_ma3', 'gt_spike', 'gt_z12', 'mom_log_12'] …


In [8]:
# Check the split and dates are correct
print(df.groupby("split")["date"].agg(["min","max","count"]))
print("Win-rate by split:\n", df.groupby("split")["label_win_12"].mean().round(3))

# Remove features that have low coverage <5%
train = df[df["split"]=="train"].copy()

# % coverage in train
coverage = train[feature_cols].notna().mean().sort_values()

# Identify very sparse features (e.g., ≤5% present) to drop
low_cov = coverage[coverage <= 0.05].index.tolist()
print("Low-coverage features:", low_cov) 
if low_cov:
    print("Dropping low-coverage features:", list(low_cov))
    feature_cols = [c for c in feature_cols if c not in low_cov]

             min        max  count
split                             
test  2022-01-31 2024-09-30    330
train 2005-02-28 2017-12-31   1550
val   2018-01-31 2021-12-31    480
Win-rate by split:
 split
test     0.594
train    0.515
val      0.523
Name: label_win_12, dtype: float64
Low-coverage features: []


# ORIGINAL APPROACH 

In [24]:
# Logisitic Regression

val   = df[df["split"]=="val"].copy()
test  = df[df["split"]=="test"].copy()

X_train, y_train = train[feature_cols], train["label_win_12"].astype(int)
X_val,   y_val   = val[feature_cols],   val["label_win_12"].astype(int)
X_test,  y_test  = test[feature_cols],  test["label_win_12"].astype(int)


clf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("model",   LogisticRegression(max_iter=1000, solver="lbfgs"))  # add class_weight="balanced" if needed
])

clf.fit(X_train, y_train)


print("Train accuracy:", clf.score(X_train, y_train))
print("Val accuracy:",   clf.score(X_val, y_val))

# F1
p_val = clf.predict_proba(X_val)[:, 1]
candidates = np.linspace(0.20, 0.80, 61)

best_t, best_f1 = 0.5, -1
for t in candidates:
    yhat = (p_val >= t).astype(int)
    f1 = f1_score(y_val, yhat, zero_division=0)
    if f1 > best_f1:
        best_f1, best_t = f1, t

print(f"Best validation threshold by F1: {best_t:.2f} (F1={best_f1:.3f})")
print("Val AUC (threshold-free):", roc_auc_score(y_val, p_val))


# Balanced 
p_val = clf.predict_proba(X_val)[:, 1]
cands = np.linspace(0.20, 0.80, 121)

# Option A: maximise Balanced Accuracy = (TPR + TNR)/2
best_t, best_balacc = 0.5, -1.0
for t in cands:
    yhat = (p_val >= t).astype(int)
    bal = balanced_accuracy_score(y_val, yhat)
    if bal > best_balacc:
        best_balacc, best_t = bal, t

print(f"Best threshold by Balanced ACC: {best_t:.2f} (BalACC={best_balacc:.3f})")
print("Val AUC (threshold-free):", roc_auc_score(y_val, p_val))

# evaluate at that fixed threshold
eval_split("Validation @best_t(BalACC)", clf, X_val,  y_val,  thresh=best_t)
eval_split("Test @best_t(BalACC)",      clf, X_test, y_test, thresh=best_t)


Train accuracy: 0.6251612903225806
Val accuracy: 0.5041666666666667
Best validation threshold by F1: 0.29 (F1=0.700)
Val AUC (threshold-free): 0.557786321961064
Best threshold by Balanced ACC: 0.67 (BalACC=0.584)
Val AUC (threshold-free): 0.557786321961064

=== Validation @best_t(BalACC) ===
AUC: 0.558 | ACC: 0.573 | PREC: 0.685 | REC: 0.339 | F1: 0.453
              precision    recall  f1-score   support

           0      0.534     0.830     0.650       229
           1      0.685     0.339     0.453       251

    accuracy                          0.573       480
   macro avg      0.610     0.584     0.551       480
weighted avg      0.613     0.573     0.547       480


=== Test @best_t(BalACC) ===
AUC: 0.412 | ACC: 0.400 | PREC: 0.490 | REC: 0.245 | F1: 0.327
              precision    recall  f1-score   support

           0      0.362     0.627     0.459       134
           1      0.490     0.245     0.327       196

    accuracy                          0.400       330
   mac

In [25]:
def auc_by_year(model, X, y, dates):
    d = pd.DataFrame({"date": dates, "y": y, "p": model.predict_proba(X)[:,1]})
    d["year"] = d["date"].dt.year
    return d.groupby("year").apply(lambda g: roc_auc_score(g["y"], g["p"]) if g["y"].nunique()==2 else float("nan"))

print("Val AUC by year:\n", auc_by_year(clf, X_val, y_val, df.loc[X_val.index, "date"]))
print("Test AUC by year:\n", auc_by_year(clf, X_test, y_test, df.loc[X_test.index, "date"]))


Val AUC by year:
 year
2018    0.590086
2019    0.342427
2020    0.763853
2021    0.567373
dtype: float64
Test AUC by year:
 year
2022    0.562689
2023    0.405113
2024    0.247500
dtype: float64


/var/folders/m2/10p9b8xj3gg5xtr6q8r54cs40000gn/T/ipykernel_94674/1691257736.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return d.groupby("year").apply(lambda g: roc_auc_score(g["y"], g["p"]) if g["y"].nunique()==2 else float("nan"))
/var/folders/m2/10p9b8xj3gg5xtr6q8r54cs40000gn/T/ipykernel_94674/1691257736.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return d.groupby("year").apply(lambda g: roc_auc_sc

In [23]:
def eval_split(name, model, X, y, thresh=0.5):
    p = model.predict_proba(X)[:,1]
    yhat = (p >= thresh).astype(int)
    auc = roc_auc_score(y, p)
    acc = accuracy_score(y, yhat)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y, yhat, average="binary", zero_division=0
    )
    print(f"\n=== {name} ===")
    print(f"AUC: {auc:.3f} | ACC: {acc:.3f} | PREC: {prec:.3f} | REC: {rec:.3f} | F1: {f1:.3f}")
    print(classification_report(y, yhat, digits=3))

eval_split("Train",      clf, X_train, y_train)
eval_split("Validation", clf, X_val,   y_val)
eval_split("Test",       clf, X_test,  y_test)


=== Train ===
AUC: 0.683 | ACC: 0.625 | PREC: 0.642 | REC: 0.615 | F1: 0.628
              precision    recall  f1-score   support

           0      0.609     0.636     0.622       752
           1      0.642     0.615     0.628       798

    accuracy                          0.625      1550
   macro avg      0.625     0.625     0.625      1550
weighted avg      0.626     0.625     0.625      1550


=== Validation ===
AUC: 0.558 | ACC: 0.504 | PREC: 0.521 | REC: 0.629 | F1: 0.570
              precision    recall  f1-score   support

           0      0.475     0.367     0.414       229
           1      0.521     0.629     0.570       251

    accuracy                          0.504       480
   macro avg      0.498     0.498     0.492       480
weighted avg      0.499     0.504     0.496       480


=== Test ===
AUC: 0.412 | ACC: 0.485 | PREC: 0.557 | REC: 0.648 | F1: 0.599
              precision    recall  f1-score   support

           0      0.324     0.246     0.280       134

# REGIME FLAGS FIX

In [ ]:
# =============================================================================
# REGIME SENSITIVITY FIXES
# =============================================================================
print("\n" + "="*70)
print("IMPLEMENTING REGIME FLAGS TO FIX 2023-2024 PERFORMANCE")
print("="*70)

# 60 months == 5 years 
WIN = 60

# Function to help find the rolling percentile and median
def roll_q(s, q=0.8):
    return s.rolling(WIN, min_periods=36).quantile(q)

def med(s):
    return s.rolling(WIN, min_periods=36).median()

# ---- Core percentile-based regimes ----
print("Adding adaptive regime flags...")
df[vix_p80] = roll_q(df['vix_lvl'], 0.80)
df['high_vix'] = (df["vix_lvl"] >= df["vix_p80"]).astype("int8")

# Rising rates: 10Y change over last 3 months 
df["us10y_d3m"] = df["us10y_lvl"] - df["us10y_lvl"].shift(3)
df["us10y_d3m_thr"] = roll_med(df["us10y_d3m"])
df["rising_rates"]  = (df["us10y_d3m"] > df["us10y_d3m_thr"]).astype("int8")

# Oil shock: large absolute 3m Brent move
abs_brent = df["brent_d3m"].abs()
df["abs_brent_p80"] = roll_q(abs_brent, 0.80)
df["oil_shock"] = (abs_brent >= df["abs_brent_p80"]).astype("int8")

# Strong dollar: DXY high vs trailing p80
df["dxy_p80"]       = roll_q(df["dxy_lvl"], 0.80)
df["strong_dollar"] = (df["dxy_lvl"] >= df["dxy_p80"]).astype("int8")

# Bear tape (market downtrend) ----
# Per-ticker benchmark momentum over prior 12m using t-1 info:
bench_12m_by_row = (
    df.groupby("ticker")["l_bench"]
      .transform(lambda s: s.shift(1).rolling(12, min_periods=12).sum())
)

# Collapse to one regime flag per month: median across your 10 names
bear_by_month = (pd.DataFrame({"date": df["date"], "bench_12m": bench_12m_by_row})
                 .groupby("date")["bench_12m"].median()
                 .lt(0).astype("int8"))

df = df.merge(bear_by_month.rename("bear_tape"), left_on="date", right_index=True, how="left")

